# ReServe memory supplement — FP16 and AWQ

This updates your memory supplement to **AWQ W4A16**, using the same pinned stack and checkpoints as `awq_profiling.ipynb`. It measures resident allocation, actual FP16 KV tensors, one-token cache growth, operation peaks, and memory after request cleanup. It does not repeat the 50-trial latency sweep or implement live precision switching.

Use one Kaggle Tesla T4 on GPU 0, with Internet enabled. Do not run another model in this kernel.

**Run order:** install once → restart kernel → Steps 2–5 with `MODE = "awq"` → download → restart kernel → Steps 2–5 with `MODE = "fp16"` → download. Rerun both modes: the old FP16/NF4 profiles used a different software stack.

Qwen supplies the prepared AWQ checkpoint. No calibration dataset or quantization run is needed. Actual GPU execution must pass this notebook's smoke test before its measurements can be used.

## 1. Install once, then restart

This is the same installation cell as the timing notebook: Torch 2.6.0 + CUDA 12.4, Transformers 4.51.3, AutoAWQ 0.2.9, and Triton 3.2.0. It replaces this notebook environment's packages. Use the same stack for both precision modes. After installation, restart the Python kernel and start at Step 2.

In [ ]:
import subprocess
import sys

if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError("Use a Kaggle Python 3.10-3.13 kernel for this pinned profile environment")

# Install the documented AutoAWQ-compatible torch stack, including matching
# vision/audio packages so stale Kaggle extensions do not break model imports.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "torch==2.6.0", "triton==3.2.0", "autoawq==0.2.9",
    "transformers==4.51.3", "accelerate==1.6.0", "huggingface-hub==0.30.2",
    "tokenizers==0.21.1", "safetensors==0.5.3", "numpy==2.2.6", "pandas==2.2.3",
    "datasets==3.5.0", "librosa==0.11.0",
])
print("Installation finished. RESTART THE PYTHON KERNEL now.")
print("After restarting, start at Step 2; do not rerun this installation cell.")

## 2. Select a mode

Run AWQ first. After downloading its ZIP, restart the kernel and change `MODE` to `"fp16"`. The default grid contains 16 shapes, with three fresh request lifecycles per shape (48 trials per mode). Checkpoint commits are defined explicitly in the helper cell; `MODEL_REVISION = None` selects the pinned commit for the chosen mode.

In [ ]:
MODE = "awq"                  # "awq" first; restart, then "fp16"
MODEL_REVISION = None           # Select the mode-specific pinned commit
EXPECTED_GPU = "Tesla T4"
BATCH_SIZES = [1, 2, 4, 8]
LENGTHS = [256, 512, 1024, 2048]
MEMORY_REPEATS = 3

## 3. Define the measurement code

This cell only defines helpers. Step 4 loads one model and checks actual packed AWQ projections, FP16 KV shape/dtype, finite logits, and exact one-token cache growth. Fusion is disabled to preserve layer/cache inspection.

Both parameters and buffers are inventoried. AWQ's packed weights are buffers, so summing parameters alone would undercount its memory. Tied backing storage is counted once. The loader deliberately uses Triton, refusing a naive slow fallback.

In [ ]:
"""Shared, embedded helpers for ReServe's fixed FP16/AWQ notebook profiles."""
import gc
import itertools
import json
import os
import platform
import random
import re
import statistics
import time
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import pandas as pd
import torch
import transformers
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, AwqConfig

MODEL_IDS = {
    "fp16": "Qwen/Qwen2.5-1.5B-Instruct",
    "awq": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
}
MODEL_REVISIONS = {
    "fp16": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    "awq": "c5b72040769bfde6c5b562ae890f9657b0803f79",
}
PACKAGE_PINS = {
    "torch": "2.6.0", "transformers": "4.51.3", "autoawq": "0.2.9",
    "triton": "3.2.0", "accelerate": "1.6.0", "huggingface-hub": "0.30.2",
    "numpy": "2.2.6", "pandas": "2.2.3",
}
INPUT_TOKEN_ID = 14990
EXPECTED_KV_BYTES_PER_TOKEN = 28672


def prepare_environment(expected_gpu="Tesla T4"):
    actual_packages = {}
    for name, expected in PACKAGE_PINS.items():
        try:
            actual_packages[name] = version(name)
        except PackageNotFoundError:
            actual_packages[name] = "missing"
    wrong = {name: (actual_packages[name], expected) for name, expected in PACKAGE_PINS.items()
             if actual_packages[name].split("+")[0] != expected}
    if wrong or transformers.__version__ != PACKAGE_PINS["transformers"] or torch.__version__.split("+")[0] != PACKAGE_PINS["torch"]:
        raise RuntimeError(
            f"Pinned AWQ environment is not active: {wrong}. Run the installation cell, "
            "restart the Python kernel, then run from the settings cell."
        )
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle NVIDIA GPU. This profiler requires CUDA.")
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
    gpu = torch.cuda.get_device_name(device)
    if expected_gpu is not None and gpu != expected_gpu:
        raise RuntimeError(f"Expected {expected_gpu}, got {gpu}. Use the same GPU for both modes.")
    if tuple(torch.cuda.get_device_capability(device)) < (7, 5):
        raise RuntimeError("This AWQ configuration targets T4-class compute capability 7.5 or newer.")
    if torch.version.cuda != "12.4":
        raise RuntimeError(f"Expected the cu124 wheel, got CUDA runtime {torch.version.cuda}.")
    torch.manual_seed(42)
    random.seed(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.cuda.synchronize(device)
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.memory_allocated(device) != 0:
        raise RuntimeError("GPU tensors already exist. Restart the kernel before profiling another mode.")
    return device, {
        "gpu": gpu, "compute_capability": list(torch.cuda.get_device_capability(device)),
        "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
        "transformers": transformers.__version__, "python": platform.python_version(),
        "packages": actual_packages,
    }


def cache_pairs(cache):
    """Read both older key_cache/value_cache and newer .layers cache layouts."""
    if cache is None:
        raise ValueError("Model did not return a KV cache")
    if hasattr(cache, "layers"):
        return [(layer.keys, layer.values) for layer in cache.layers]
    if hasattr(cache, "key_cache") and hasattr(cache, "value_cache"):
        if len(cache.key_cache) != len(cache.value_cache):
            raise ValueError("Unequal key/value cache layer counts")
        return list(zip(cache.key_cache, cache.value_cache))
    if isinstance(cache, (tuple, list)):
        if any(len(layer) != 2 for layer in cache):
            raise ValueError("Expected exactly keys and values in each legacy cache layer")
        return [(layer[0], layer[1]) for layer in cache]
    raise ValueError(f"Unsupported cache type: {type(cache).__name__}")


def inspect_cache(cache, batch_size, length, config):
    pairs = cache_pairs(cache)
    if len(pairs) != config.num_hidden_layers:
        raise ValueError(f"Expected {config.num_hidden_layers} KV layers, got {len(pairs)}")
    head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
    expected_shape = (batch_size, config.num_key_value_heads, length, head_dim)
    logical_bytes, unique_storages, details = 0, {}, []
    for index, pair in enumerate(pairs):
        for kind, tensor in zip(("keys", "values"), pair):
            if tuple(tensor.shape) != expected_shape:
                raise ValueError(f"Layer {index} {kind}: {tuple(tensor.shape)} != {expected_shape}")
            if tensor.dtype != torch.float16 or str(tensor.device) != "cuda:0":
                raise ValueError(f"Expected FP16 KV on cuda:0; got {tensor.dtype}, {tensor.device}")
            logical = int(tensor.numel() * tensor.element_size())
            storage = tensor.untyped_storage()
            key = (str(tensor.device), int(storage.data_ptr()))
            backing = int(storage.nbytes())
            seen = key in unique_storages
            unique_storages[key] = backing
            logical_bytes += logical
            details.append({
                "layer": index, "kind": kind, "shape": json.dumps(list(tensor.shape)),
                "dtype": str(tensor.dtype), "logical_bytes": logical,
                "backing_storage_bytes": backing, "storage_seen_previously": seen,
            })
    expected = 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * 2 * batch_size * length
    if logical_bytes != expected:
        raise ValueError(f"KV size {logical_bytes} != architecture-derived {expected}")
    return {
        "kv_logical_bytes": logical_bytes, "kv_unique_backing_bytes": sum(unique_storages.values()),
        "kv_unique_storages": len(unique_storages),
        "kv_bytes_per_cached_token": logical_bytes // (batch_size * length),
        "kv_dtype": "float16", "cached_tokens_per_request": length,
    }, details


def model_inventory(model):
    """Include packed AWQ buffers; count tied/shared backing storage once."""
    seen, rows, layers = set(), [], {}
    for role, iterator in (
        ("parameter", model.named_parameters(remove_duplicate=False)),
        ("buffer", model.named_buffers(remove_duplicate=False)),
    ):
        for name, tensor in iterator:
            match = re.match(r"^model\.layers\.(\d+)\.", name)
            layer = int(match.group(1)) if match else -1
            storage = tensor.untyped_storage()
            key = (str(tensor.device), int(storage.data_ptr()), int(storage.nbytes()))
            unique = key not in seen
            seen.add(key)
            charged = int(storage.nbytes()) if unique else 0
            rows.append({
                "tensor_name": name, "role": role, "decoder_layer": layer,
                "dtype": str(tensor.dtype), "shape": json.dumps(list(tensor.shape)),
                "numel": int(tensor.numel()), "logical_bytes": int(tensor.numel() * tensor.element_size()),
                "backing_storage_bytes": int(storage.nbytes()), "unique_backing_bytes": charged,
                "shared_storage": not unique,
            })
            item = layers.setdefault(layer, {"decoder_layer": layer, "unique_backing_bytes": 0,
                                             "parameter_backing_bytes": 0, "buffer_backing_bytes": 0,
                                             "awq_linear_count": 0})
            item["unique_backing_bytes"] += charged
            item[role + "_backing_bytes"] += charged
    for name, module in model.named_modules():
        if all(hasattr(module, attr) for attr in ("qweight", "qzeros", "scales")):
            match = re.match(r"^model\.layers\.(\d+)\.", name)
            layer = int(match.group(1)) if match else -1
            layers[layer]["awq_linear_count"] += 1
    return rows, [layers[key] for key in sorted(layers)]


def load_profile_model(mode, revision=None):
    if mode not in MODEL_IDS:
        raise ValueError('MODE must be "fp16" or "awq"')
    model_id = MODEL_IDS[mode]
    revision = revision or MODEL_REVISIONS[mode]
    config = AutoConfig.from_pretrained(model_id, revision=revision, trust_remote_code=False)
    resolved = getattr(config, "_commit_hash", None)
    if resolved != revision:
        raise ValueError(f"Requested commit {revision}, config resolved to {resolved}")
    fields = ("model_type", "hidden_size", "intermediate_size", "num_hidden_layers",
              "num_attention_heads", "num_key_value_heads", "vocab_size", "tie_word_embeddings",
              "max_position_embeddings", "rope_theta", "rms_norm_eps")
    if mode == "awq":
        base_config = AutoConfig.from_pretrained(MODEL_IDS["fp16"], revision=MODEL_REVISIONS["fp16"], trust_remote_code=False)
        mismatches = {key: (getattr(config, key, None), getattr(base_config, key, None))
                      for key in fields if getattr(config, key, None) != getattr(base_config, key, None)}
        if mismatches:
            raise ValueError(f"FP16/AWQ architecture mismatch: {mismatches}")
        quant = getattr(config, "quantization_config", {})
        if hasattr(quant, "to_dict"):
            quant = quant.to_dict()
        for key, expected in {"quant_method": "awq", "bits": 4, "group_size": 128,
                              "zero_point": True, "version": "gemm"}.items():
            if quant.get(key) != expected:
                raise ValueError(f"AWQ checkpoint {key}={quant.get(key)!r}; expected {expected!r}")
        from awq.modules.linear import gemm as awq_gemm
        # Deliberately choose Triton: no torch-ABI-specific autoawq-kernels wheel.
        awq_gemm.awq_ext = None
        if not awq_gemm.TRITON_AVAILABLE:
            raise RuntimeError("AWQ Triton kernels unavailable; refusing a naive slow fallback")
        options = {"quantization_config": AwqConfig(
            bits=4, group_size=128, zero_point=True, version="gemm", backend="autoawq",
            do_fuse=False, modules_to_not_convert=["lm_head"],
        )}
    else:
        options = {}
        if getattr(config, "quantization_config", None):
            raise ValueError("FP16 source unexpectedly contains quantization metadata")
    model = AutoModelForCausalLM.from_pretrained(
        model_id, revision=revision, config=config, torch_dtype=torch.float16,
        device_map={"": 0}, attn_implementation="sdpa", low_cpu_mem_usage=True,
        trust_remote_code=False, use_safetensors=True, **options,
    ).eval()
    if any(str(t.device) != "cuda:0" for t in itertools.chain(model.parameters(), model.buffers())):
        raise ValueError("Every parameter and buffer must reside on GPU 0; no CPU/disk offload")
    if any(p.dtype != torch.float16 for p in model.parameters() if p.is_floating_point()):
        raise ValueError("Floating parameters must use FP16")
    quantized = [(name, module) for name, module in model.named_modules()
                 if all(hasattr(module, attr) for attr in ("qweight", "qzeros", "scales"))]
    if mode == "awq":
        from awq.modules.linear.gemm import WQLinear_GEMM
        if len(quantized) != 196 or any(not isinstance(m, WQLinear_GEMM) for _, m in quantized):
            raise ValueError(f"Expected 196 unfused AWQ GEMM linear modules, got {len(quantized)}")
        for name, module in quantized:
            if module.w_bit != 4 or module.group_size != 128:
                raise ValueError(f"Wrong AWQ format in {name}")
            if module.qweight.dtype != torch.int32 or module.qzeros.dtype != torch.int32 or module.scales.dtype != torch.float16:
                raise ValueError(f"Wrong packed-weight/metadata dtypes in {name}")
        if getattr(model, "_awq_is_fused", False):
            raise ValueError("Attention/MLP fusion must remain disabled for cache and layer-level inspection")
    elif quantized or getattr(model, "is_quantized", False):
        raise ValueError("FP16 mode unexpectedly contains quantized modules")
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
    if tokenizer.encode("hello", add_special_tokens=False) != [INPUT_TOKEN_ID]:
        raise ValueError("Tokenizer differs from the original profiling input")
    inventory, layer_bytes = model_inventory(model)
    if mode == "awq" and any(row["awq_linear_count"] != 7 for row in layer_bytes if row["decoder_layer"] >= 0):
        raise ValueError("Each of the 28 decoder layers must contain seven AWQ projections")
    quant = getattr(model.config, "quantization_config", None)
    identity = {
        "model_id": model_id, "model_revision": revision, "resolved_config_revision": resolved,
        "fp16_reference_model_id": MODEL_IDS["fp16"], "fp16_reference_revision": MODEL_REVISIONS["fp16"],
        "quantization_config": quant.to_dict() if hasattr(quant, "to_dict") else quant,
        "quantized_linear_layers": len(quantized), "model_config": model.config.to_dict(),
        "awq_kernel_backend": "autoawq_0.2.9_gemm_triton" if mode == "awq" else None,
        "awq_fusion_enabled": False, "attention_implementation_requested": "sdpa",
        "attention_implementation_actual": model.config._attn_implementation,
        "resident_tensor_unique_backing_bytes": sum(row["unique_backing_bytes"] for row in inventory),
        "comparison_note": "Rerun FP16 on this same pinned stack. Earlier 2.11/5.20 profiles are historical only.",
        "checkpoint_lineage_note": "Official FP16 and official prequantized AWQ are separately pinned repositories; identical quantization-source FP16 commit is not independently established.",
        "awq_execution_note": "AutoAWQ GEMM uses Triton packed GEMM for small token batches and dequantize+FP16 matmul for batch*sequence >=1024. Workspace/timing include this backend heuristic.",
        "layer_variant_note": "Inventory includes all layer parameters and buffers, including FP16 tensors whose values may have been rescaled during AWQ preparation.",
    }
    return model, identity, inventory, layer_bytes


def write_inventory(directory, inventory, layer_bytes):
    pd.DataFrame(inventory).to_csv(directory / "model_tensors.csv", index=False)
    pd.DataFrame(layer_bytes).to_csv(directory / "layer_memory.csv", index=False)


@torch.inference_mode()
def smoke_check(model, device):
    ids = mask = output = cache = token = decoded = decode_mask = None
    try:
        ids = torch.full((1, 16), INPUT_TOKEN_ID, dtype=torch.long, device=device)
        mask = torch.ones_like(ids)
        output = model(input_ids=ids, attention_mask=mask, use_cache=True, logits_to_keep=1)
        cache = output.past_key_values
        first, _ = inspect_cache(cache, 1, 16, model.config)
        token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        if not torch.isfinite(output.logits).all().item():
            raise ValueError("Non-finite prefill logits")
        output = None
        decode_mask = torch.ones((1, 17), dtype=torch.long, device=device)
        decoded = model(input_ids=token, attention_mask=decode_mask, past_key_values=cache,
                        use_cache=True, logits_to_keep=1)
        second, _ = inspect_cache(decoded.past_key_values, 1, 17, model.config)
        if not torch.isfinite(decoded.logits).all().item():
            raise ValueError("Non-finite decode logits")
        if first["kv_bytes_per_cached_token"] != EXPECTED_KV_BYTES_PER_TOKEN:
            raise ValueError("Unexpected KV bytes per cached token")
        if second["kv_logical_bytes"] - first["kv_logical_bytes"] != EXPECTED_KV_BYTES_PER_TOKEN:
            raise ValueError("Incorrect one-token cache growth")
        return {"kv_dtype_verified": "float16", "kv_bytes_per_token_verified": EXPECTED_KV_BYTES_PER_TOKEN,
                "cache_type": type(cache).__name__, "smoke_test": "passed"}
    finally:
        ids = mask = output = cache = token = decoded = decode_mask = None
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize(device)


def host_memory():
    """CPU RSS and available system RAM, in bytes; optional Linux observations."""
    result = {"host_rss_bytes": None, "host_available_bytes": None}
    for filename, field, target in [
        ("/proc/self/status", "VmRSS:", "host_rss_bytes"),
        ("/proc/meminfo", "MemAvailable:", "host_available_bytes"),
    ]:
        try:
            for line in Path(filename).read_text().splitlines():
                if line.startswith(field):
                    result[target] = int(line.split()[1]) * 1024
                    break
        except (OSError, ValueError):
            pass
    return result


def observe(device):
    """Synchronized scalars only: never return a live GPU tensor."""
    torch.cuda.synchronize(device)
    free, total = torch.cuda.mem_get_info(device)
    allocated = int(torch.cuda.memory_allocated(device))
    reserved = int(torch.cuda.memory_reserved(device))
    return {
        "allocated_bytes": allocated,
        "reserved_bytes": reserved,
        "peak_allocated_bytes": int(torch.cuda.max_memory_allocated(device)),
        "peak_reserved_bytes": int(torch.cuda.max_memory_reserved(device)),
        "device_free_bytes": int(free),
        "device_total_bytes": int(total),
        "device_global_used_bytes": int(total - free),
        # Diagnostic only: includes the CUDA context and other allocations/users.
        "device_used_minus_torch_reserved_bytes": int(total - free - reserved),
        **host_memory(),
    }


def check_growth(before, after, batch_size, bytes_per_token):
    growth = after["kv_logical_bytes"] - before["kv_logical_bytes"]
    expected = batch_size * bytes_per_token
    if growth != expected:
        raise ValueError(f"One decode step grew KV by {growth} bytes; expected {expected}")
    return growth


def non_kv_upper_bound(peak, baseline, minimum_live_kv):
    """Loose bound from this trial, NOT an isolated workspace measurement."""
    return max(0, int(peak) - int(baseline) - int(minimum_live_kv))


def write_checkpoint(directory, metadata, snapshots, trials, cache_details):
    """Atomic file replacements keep earlier measurements on failure/interruption."""
    frames = {
        "memory_snapshots.csv": snapshots,
        "memory_trials.csv": trials,
        "cache_tensors.csv": cache_details,
    }
    for name, rows in frames.items():
        if not rows:
            continue
        temporary = directory / (name + ".tmp")
        pd.DataFrame(rows).to_csv(temporary, index=False)
        temporary.replace(directory / name)
    temporary = directory / "metadata.json.tmp"
    temporary.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    temporary.replace(directory / "metadata.json")


@torch.inference_mode()
def warmup_model(model, device, batch_size, length, token_id):
    """Initialize representative kernels; leave no request/cache references."""
    ids = mask = prefix = cache = token = decode_mask = decoded = None
    try:
        ids = torch.full((batch_size, length), token_id, dtype=torch.long, device=device)
        mask = torch.ones_like(ids)
        prefix = model(input_ids=ids, attention_mask=mask, use_cache=True, logits_to_keep=1)
        cache = prefix.past_key_values
        inspect_cache(cache, batch_size, length, model.config)
        token = prefix.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        decode_mask = torch.ones((batch_size, length + 1), dtype=torch.long, device=device)
        decoded = model(input_ids=token, attention_mask=decode_mask,
                        past_key_values=cache, use_cache=True, logits_to_keep=1)
        inspect_cache(decoded.past_key_values, batch_size, length + 1, model.config)
        if not torch.isfinite(prefix.logits).all().item() or not torch.isfinite(decoded.logits).all().item():
            raise ValueError("Warm-up produced non-finite logits")
        torch.cuda.synchronize(device)
    finally:
        ids = mask = prefix = cache = token = decode_mask = decoded = None
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize(device)


@torch.inference_mode()
def measure_request(model, device, batch_size, length, token_id, identity,
                    snapshots, cache_details, baseline, bytes_per_token, tolerance):
    """One fresh prompt, ONE cached decode step, then complete request release."""
    ids = prompt_mask = decode_mask = prefix = cache = next_token = decoded = None
    row = {**identity, "status": "started", "last_stage": "baseline"}

    def snap(stage, kv=None, peak_scope="none"):
        observation = observe(device)
        snapshots.append({**identity, "stage": stage, "peak_scope": peak_scope,
                          **observation, **(kv or {})})
        row["last_stage"] = stage
        return observation

    def record_cache(stage, current_cache, cached_length):
        info, details = inspect_cache(current_cache, batch_size, cached_length, model.config)
        cache_details.extend({**identity, "stage": stage, **detail} for detail in details)
        return info

    try:
        ids = torch.full((batch_size, length), token_id, dtype=torch.long, device=device)
        prompt_mask = torch.ones_like(ids)
        decode_mask = torch.ones((batch_size, length + 1), dtype=torch.long, device=device)
        inputs_ready = snap("inputs_ready")
        torch.cuda.reset_peak_memory_stats(device)
        prefix = model(input_ids=ids, attention_mask=prompt_mask, use_cache=True, logits_to_keep=1)
        prefill_peak = snap("prefill_returned", peak_scope="prefill")
        cache = prefix.past_key_values
        prefill_kv = record_cache("after_prefill", cache, length)

        next_token = prefix.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        prefix = None
        before_decode = snap("cache_retained_before_decode", prefill_kv)
        torch.cuda.reset_peak_memory_stats(device)
        decoded = model(input_ids=next_token, attention_mask=decode_mask,
                        past_key_values=cache, use_cache=True, logits_to_keep=1)
        decode_peak = snap("decode_returned", peak_scope="decode")
        cache = decoded.past_key_values
        decode_kv = record_cache("after_decode", cache, length + 1)
        growth = check_growth(prefill_kv, decode_kv, batch_size, bytes_per_token)

        # Retain ONLY the cache, with model weights still resident.
        ids = prompt_mask = decode_mask = next_token = decoded = None
        kv_only = snap("only_cache_retained", decode_kv)
        row.update({
            "baseline_allocated_bytes": baseline["allocated_bytes"],
            "baseline_reserved_bytes": baseline["reserved_bytes"],
            "inputs_ready_allocated_bytes": inputs_ready["allocated_bytes"],
            "prefill_peak_allocated_bytes": prefill_peak["peak_allocated_bytes"],
            "prefill_peak_reserved_bytes": prefill_peak["peak_reserved_bytes"],
            "prefill_return_allocated_bytes": prefill_peak["allocated_bytes"],
            "prefill_kv_logical_bytes": prefill_kv["kv_logical_bytes"],
            "prefill_kv_unique_backing_bytes": prefill_kv["kv_unique_backing_bytes"],
            "before_decode_allocated_bytes": before_decode["allocated_bytes"],
            "decode_peak_allocated_bytes": decode_peak["peak_allocated_bytes"],
            "decode_peak_reserved_bytes": decode_peak["peak_reserved_bytes"],
            "decode_return_allocated_bytes": decode_peak["allocated_bytes"],
            "decode_kv_logical_bytes": decode_kv["kv_logical_bytes"],
            "decode_kv_unique_backing_bytes": decode_kv["kv_unique_backing_bytes"],
            "kv_bytes_per_cached_token": decode_kv["kv_bytes_per_cached_token"],
            "decode_growth_bytes": growth,
            "kv_only_allocated_bytes": kv_only["allocated_bytes"],
            # Prefill starts with zero KV; decode starts with the existing prompt KV.
            # These upper bounds include inputs/logits/allocator overhead. Prefill's
            # bound is particularly loose because KV itself grows during execution.
            "prefill_non_kv_upper_bound_bytes": non_kv_upper_bound(
                prefill_peak["peak_allocated_bytes"], baseline["allocated_bytes"], 0),
            "decode_non_kv_upper_bound_bytes": non_kv_upper_bound(
                decode_peak["peak_allocated_bytes"], baseline["allocated_bytes"],
                prefill_kv["kv_logical_bytes"]),
            "status": "ok",
        })
    except BaseException as exc:
        row["failure_stage"] = row["last_stage"]
        row["status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else (
            "oom" if isinstance(exc, torch.cuda.OutOfMemoryError) else "error")
        row["error"] = f"{type(exc).__name__}: {str(exc).splitlines()[0] if str(exc) else ''}"
        raise
    finally:
        # No tensor or exception object is stored in the result records.
        ids = prompt_mask = decode_mask = prefix = cache = next_token = decoded = None
        gc.collect()
        released = snap("request_released_pool_kept")
        torch.cuda.empty_cache()
        trimmed = snap("request_released_pool_trimmed")
        difference = released["allocated_bytes"] - baseline["allocated_bytes"]
        row.update({
            "released_allocated_bytes": released["allocated_bytes"],
            "released_reserved_bytes": released["reserved_bytes"],
            "trimmed_allocated_bytes": trimmed["allocated_bytes"],
            "trimmed_reserved_bytes": trimmed["reserved_bytes"],
            "released_minus_baseline_bytes": difference,
            "cleanup_within_tolerance": abs(difference) <= tolerance,
        })
        # Caller receives a scalar checkpoint even if an exception is re-raised.
        identity["_trial_result"] = row
    return row


def build_summaries(trials):
    """Keep one row per supported shape; never subtract unrelated maxima."""
    groups = {}
    for row in trials:
        groups.setdefault((row["batch_size"], row["length_tokens"]), []).append(row)
    result = []
    for (batch_size, length), group in sorted(groups.items()):
        ok = [r for r in group if r["status"] == "ok"]
        item = {"batch_size": batch_size, "length_tokens": length,
                "trials_recorded": len(group), "ok_trials": len(ok),
                "status": "ok" if len(ok) == len(group) else "incomplete_or_error"}
        if ok:
            for column in (
                "baseline_allocated_bytes", "prefill_peak_allocated_bytes",
                "decode_peak_allocated_bytes", "prefill_kv_logical_bytes",
                "decode_kv_logical_bytes", "prefill_non_kv_upper_bound_bytes",
                "decode_non_kv_upper_bound_bytes",
            ):
                item[column + "_max"] = max(r[column] for r in ok)
            item["baseline_allocated_bytes_median"] = statistics.median(r["baseline_allocated_bytes"] for r in ok)
            item["max_absolute_cleanup_delta_bytes"] = max(abs(r["released_minus_baseline_bytes"]) for r in ok)
            item["all_cleanup_checks_pass"] = all(r["cleanup_within_tolerance"] for r in ok)
        result.append(item)
    return result




def run_memory_supplement(mode, model_revision, batches, lengths, repeats=3):
    """One fresh-kernel mode per invocation; no latency sweep or live switches."""
    if mode not in ("fp16", "awq"):
        raise ValueError('MODE must be "fp16" or "awq"')
    if not batches or not lengths or repeats < 1:
        raise ValueError("Choose nonempty positive batch/length lists and repeats >= 1")
    if any(not isinstance(x, int) or isinstance(x, bool) or x <= 0 for x in batches + lengths):
        raise ValueError("Batch sizes and lengths must be positive integers")
    if len(set(batches)) != len(batches) or len(set(lengths)) != len(lengths):
        raise ValueError("Duplicate batch sizes or lengths")
    device, actual = prepare_environment(EXPECTED_GPU)
    model_id = MODEL_IDS[mode]
    model_revision = model_revision or MODEL_REVISIONS[mode]
    token_id = INPUT_TOKEN_ID
    run_id = f"memory_{mode}_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    directory = Path("/kaggle/working/reserve_memory") / run_id
    directory.mkdir(parents=True, exist_ok=False)
    snapshots, trials, details = [], [], []
    before_load = observe(device)
    snapshots.append({"run_id": run_id, "mode": mode, "stage": "before_model_load", **before_load})
    metadata = {
        "schema_version": 2, "run_id": run_id, "mode": mode, "run_status": "loading",
        "model_id": model_id, "requested_revision": model_revision,
        "historical_results_comparable": False,
        "revision_note": "Separate official FP16 and prequantized AWQ commits are pinned. Rerun FP16 on this same stack; old timing/memory results are historical only.",
        "pid": os.getpid(), "python": platform.python_version(), "environment": actual,
        "gpu_total_memory_bytes": torch.cuda.get_device_properties(device).total_memory,
        "packages": {name: version(name) for name in ("torch", "transformers", "autoawq", "triton", "accelerate", "pandas")},
        "allocator_environment": {name: os.environ.get(name) for name in ("PYTORCH_ALLOC_CONF", "PYTORCH_CUDA_ALLOC_CONF", "CUDA_VISIBLE_DEVICES")},
        "attention_implementation_requested": "sdpa", "batch_sizes": batches,
        "lengths": lengths, "repeats": repeats, "input_token_id": token_id,
        "cleanup_tolerance_bytes": 1024**2,
        "expected_trial_count": len(batches) * len(lengths) * repeats,
        "units": "All memory columns are integer bytes; display converts to MiB.",
        "measurement_notes": [
            "Allocator allocated and reserved are overlapping views; never add them.",
            "Cache logical bytes and unique backing storage are also overlapping views.",
            "Device usage/free memory is a whole-device observation, not exclusively this process.",
            "Resident baseline includes persistent tensors/quantization state; this is not pure weight bytes.",
            "Packed AWQ weight buffers are counted in model_tensors.csv and layer_memory.csv; parameters alone would undercount weights.",
            "Cache inspection returns scalars; it does not intentionally keep cache tensors alive.",
            "Operation peaks are taken in the same trial as its baseline and cache checks.",
            "non_kv_upper_bound subtracts only a guaranteed minimum live KV from the operation peak-minus-baseline; it is not exact workspace.",
            "The prefill bound is deliberately loose: it includes cache construction. Do not add components to an already observed total peak.",
            "HF DynamicCache is tested; no paged allocator, shared-prefix pool, weight restoration or switch is implemented here.",
            "empty_cache observation is a diagnostic after request cleanup, not an assumed per-token production operation.",
            "Passes characterize memory for supported homogeneous shapes, not a final dynamic serving capacity.",
        ],
    }
    write_checkpoint(directory, metadata, snapshots, trials, details)
    print(f"Mode: {mode}; saving to {directory}", flush=True)
    print("This is a memory pass, not another 50-trial timing sweep.", flush=True)
    model = None
    try:
        model, identity, inventory, layer_bytes = load_profile_model(mode, model_revision)
        metadata.update(identity)
        write_inventory(directory, inventory, layer_bytes)
        metadata.update(smoke_check(model, device))
        if max(lengths) + 1 > model.config.max_position_embeddings:
            raise ValueError("Context exceeds the model's maximum")
        metadata["model_config"] = model.config.to_dict()
        head_dim = getattr(model.config, "head_dim", None) or model.config.hidden_size // model.config.num_attention_heads
        bytes_per_token = 2 * model.config.num_hidden_layers * model.config.num_key_value_heads * head_dim * 2
        if bytes_per_token != 28672:
            raise ValueError(f"Unexpected architecture-derived KV size: {bytes_per_token}")
        metadata["derived_kv_bytes_per_cached_token"] = bytes_per_token
        gc.collect()
        torch.cuda.empty_cache()
        after_load = observe(device)
        snapshots.append({"run_id": run_id, "mode": mode, "stage": "after_model_load", **after_load})
        metadata["after_model_load"] = after_load
        write_checkpoint(directory, metadata, snapshots, trials, details)

        warmup_shapes = sorted({(min(batches), min(lengths)), (max(batches), max(lengths))})
        for b, length in warmup_shapes:
            print(f"Warm-up: B={b}, L={length}", flush=True)
            warmup_model(model, device, b, length, token_id)
        baseline = observe(device)
        snapshots.append({"run_id": run_id, "mode": mode, "stage": "after_warmup_no_requests", **baseline})
        metadata["steady_baseline"] = baseline
        metadata["kv_dtype_verified"] = "float16"
        metadata["kv_bytes_per_cached_token_verified"] = bytes_per_token
        metadata["warmup_shapes"] = warmup_shapes
        metadata["run_status"] = "running"
        print(f"Steady allocated baseline: {baseline['allocated_bytes'] / 1024**2:.2f} MiB", flush=True)
        write_checkpoint(directory, metadata, snapshots, trials, details)

        cases = list(itertools.product(batches, lengths))
        random.Random(42).shuffle(cases)
        metadata["case_order"] = cases
        for batch_size, length in cases:
            for repeat in range(1, repeats + 1):
                gc.collect()
                torch.cuda.empty_cache()
                case_baseline = observe(device)
                identity = {"run_id": run_id, "mode": mode, "batch_size": batch_size,
                            "length_tokens": length, "repeat": repeat}
                snapshots.append({**identity, "stage": "before_request", **case_baseline})
                recoverable_oom = False
                try:
                    measure_request(model, device, batch_size, length, token_id, identity,
                                    snapshots, details, case_baseline, bytes_per_token,
                                    metadata["cleanup_tolerance_bytes"])
                except torch.cuda.OutOfMemoryError:
                    recoverable_oom = True
                finally:
                    record = identity.pop("_trial_result", None)
                    if record is not None:
                        trials.append(record)
                    metadata["trials_recorded"] = len(trials)
                    write_checkpoint(directory, metadata, snapshots, trials, details)
                if recoverable_oom:
                    print(f"OOM: B={batch_size} L={length}; saved diagnostics, moving to next shape", flush=True)
                    break
            current = [r for r in trials if r["batch_size"] == batch_size and r["length_tokens"] == length]
            good = [r for r in current if r["status"] == "ok"]
            if good:
                peak = max(r["prefill_peak_allocated_bytes"] for r in good) / 1024**2
                clean = all(r["cleanup_within_tolerance"] for r in good)
                print(f"B={batch_size:2} L={length:4}: {len(good)}/{repeats} trials; prefill peak={peak:.2f} MiB; cleanup={clean}", flush=True)

        summary = build_summaries(trials)
        pd.DataFrame(summary).to_csv(directory / "memory_summary.csv", index=False)
        ok = [r for r in trials if r["status"] == "ok"]
        full_coverage = len(ok) == metadata["expected_trial_count"] and len(trials) == len(ok)
        cleanup_ok = bool(ok) and all(r["cleanup_within_tolerance"] for r in ok)
        metadata["all_expected_trials_ok"] = full_coverage
        metadata["all_cleanup_checks_pass"] = cleanup_ok
        metadata["run_status"] = "completed" if full_coverage and cleanup_ok else "completed_needs_review"
        metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        write_checkpoint(directory, metadata, snapshots, trials, details)
        print(f"{metadata['run_status']}: {len(ok)} successful trials", flush=True)
        print("Download the ZIP from the next cell, then restart the kernel before changing modes.", flush=True)
        return pd.DataFrame(summary), directory
    except BaseException as exc:
        metadata["run_status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else "error"
        metadata["error"] = f"{type(exc).__name__}: {str(exc)}"
        write_checkpoint(directory, metadata, snapshots, trials, details)
        print(f"Stopped; partial diagnostics are saved in {directory}", flush=True)
        raise
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()

## 4. Run the memory pass

A small smoke test runs first, followed by warm-up at the smallest and largest selected shapes. Each diagnostic trial creates a fresh prompt cache, decodes one token, retains only KV, releases the request, then inspects memory before and after trimming unused allocator-pool memory.

Results are checkpointed after every trial. Unexpected errors stop with partial diagnostics saved; OOM trials are marked explicitly. Warm-up failure also stops. Do not treat an incomplete or cleanup-failing run as calibrated capacity.

In [ ]:
memory_summary_df, memory_output_dir = run_memory_supplement(
    mode=MODE,
    model_revision=MODEL_REVISION,
    batches=BATCH_SIZES,
    lengths=LENGTHS,
    repeats=MEMORY_REPEATS,
)

display_columns = [
    "batch_size", "length_tokens", "ok_trials", "status",
    "baseline_allocated_bytes_median",
    "prefill_peak_allocated_bytes_max", "decode_peak_allocated_bytes_max",
    "max_absolute_cleanup_delta_bytes", "all_cleanup_checks_pass",
]
readable = memory_summary_df[[c for c in display_columns if c in memory_summary_df]].copy()
for column in list(readable.columns):
    if "_bytes" in column:
        readable[column] = readable[column] / 1024**2
        readable = readable.rename(columns={column: column.replace("_bytes", "_MiB")})
display(readable.round(3))
print("Results:", memory_output_dir)

## 5. Download this mode's results

Download the ZIP from Kaggle's Output/files panel. Then restart the Python kernel before running the other mode. If you discard the entire Kaggle session, download first and reinstall when starting a new session.

In [ ]:
import shutil
from pathlib import Path
zip_path = Path(shutil.make_archive(str(memory_output_dir), "zip", root_dir=memory_output_dir))
print("Download:", zip_path)
print("Bytes:", zip_path.stat().st_size)

## What to send back

Send both new memory ZIPs, one AWQ and one FP16. For a complete new baseline, also send both ZIPs from the timing notebook.

| File | Meaning |
|---|---|
| `metadata.json` | Mode, separate pinned checkpoint commits, package/GPU identity, actual AWQ kernel choice, smoke and completion checks |
| `memory_trials.csv` | Per-request baselines, operation peaks, KV growth, cleanup checks, and conservative overhead bounds |
| `memory_snapshots.csv` | Ordered allocated/reserved/device observations through each lifecycle |
| `cache_tensors.csv` | Actual KV shape, dtype, logical bytes, and backing storage per layer |
| `memory_summary.csv` | Per-shape summaries and cleanup checks |
| `model_tensors.csv` | All resident parameters and buffers; tied backing storage charged once |
| `layer_memory.csv` | Resident tensor bytes per decoder layer, including packed quantization metadata |

Allocated and reserved memory overlap; do not add them. KV logical and backing bytes also overlap. `non_kv_upper_bound` is deliberately conservative and is not an isolated workspace measurement. Transient dequantization performed by AutoAWQ's large-batch GEMM path is included in operation peaks.

These measurements use Hugging Face's dynamic cache. They do not demonstrate that empty paged KV blocks can physically back restored weights, or measure live switch/copy latency. The two fixed endpoints also do not establish intermediate-configuration costs or output quality.

The official AWQ checkpoint and FP16 reference are separately pinned; exact quantization-source FP16 commit identity is not independently established. Inventories include the whole layer state, including potentially rescaled FP16 tensors, for later layer-variant work.

Sources: [Qwen AWQ checkpoint](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-AWQ), [AutoAWQ compatibility](https://github.com/casper-hansen/AutoAWQ), [AutoAWQ GEMM implementation](https://github.com/casper-hansen/AutoAWQ/blob/v0.2.9/awq/modules/linear/gemm.py).